# 03 — Autoencoder'lar: Dense AE → LSTM-AE (Faz ML-2)

ML-1 dersleri bu notebook'un tasarımını belirledi (`docs/ML1_BULGULAR_VE_HATALAR.md`):
satır-bazlı skor yerine **pencere** bazlı; impute'a ödül vermemek için **maske-ağırlıklı kayıp**;
Q99 eşiğin yanına **POT/EVT** eşiği (H2); değerlendirme **uçuş düzeyinde**, ML-1 füzyonuyla aynı protokol.

## 1. Autoencoder matematiği — neden reconstruction?

Autoencoder, veriyi dar bir latent uzaydan geçirip geri kurmayı öğrenir:

$$z = f_\theta(x) \in \mathbb{R}^d, \qquad \hat{x} = g_\phi(z), \qquad d \ll \text{dim}(x)$$

**Bottleneck ($d$ küçük) olmazsa** model birim fonksiyonu kopyalar ve hiçbir şey öğrenmez.
Dar geçit, modeli yalnızca **train dağılımının ana manifoldunu** ezberlemeye zorlar.
Model **yalnızca normal** pencerelerle eğitildiğinden, normal dinamiği iyi geri kurar;
hiç görmediği anomali dinamiğinde geri kurma **başarısız olur** → skor:

$$s(x) = \frac{\sum_{t,j} M_{tj}\,(x_{tj} - \hat{x}_{tj})^2}{\sum_{t,j} M_{tj}}$$

Buradaki $M \in \{0,1\}^{w \times f}$ **eksiklik maskesi** (ML-1 dersi H1): eksik değerler 0-doldurulur ama
kayba **girmez** — model "imputed değeri doğru tahmin etti" diye ödüllenmez, eksiklik anomali imzasıysa
onu maske feature'ları (veri-kalitesi modülü) yakalar, reconstruction değil.

## 2. İki mimari

**Dense AE (debug basamağı)**: pencere düzleştirilir ($w \cdot f$ boyutlu vektör):
`Linear(wf→64) → ReLU → Linear(64→16) → ReLU → Linear(16→64) → ReLU → Linear(64→wf)`
Amaç pipeline'ı doğrulamak: bu model çalışıyorsa veri yükleme/ölçekleme/threshold/değerlendirme hattı doğru.

**LSTM-AE**: zamansal bağımlılığı modelleyen asıl model:
```
x (w×f) → LSTM(32) → son gizli durum h ∈ R^16 (latent)
        → h'yi w kez tekrarla (RepeatVector)
        → LSTM(32) → Linear(f)  → x̂ (w×f)
```
LSTM kapıları ($i, f, o$) uzun menzilli dinamiği taşır: komut→tepki gecikmesi, yavaş drift,
osilasyon periyodu gibi ALFA/H6 imzaları tek satırda değil **sekans şeklinde** yaşar.
ALFA'nın ~9-10 dk normal verisiyle büyük model overfit olur → küçük tutulur (32/16).

## 3. Eşik: Q99 persentil vs POT (EVT)

ML-1'de eşik val uçuşlarının maksimum skoruydu — 1-2 uçuşla kararsız (H2).
**Peaks-Over-Threshold**: yüksek bir başlangıç seviyesi $u$ (val skorlarının Q98'i) üzerindeki
aşımlara Generalized Pareto Distribution oturtulur (Pickands–Balkema–de Haan):

$$P(X - u > y \mid X > u) \approx \left(1 + \frac{\xi y}{\tilde\sigma}\right)^{-1/\xi}$$

Hedef risk $q$ (ör. $10^{-3}$: normal pencerelerin binde biri) için eşik:

$$\tau_{POT} = u + \frac{\tilde\sigma}{\xi}\left(\left(\frac{q\,n}{N_u}\right)^{-\xi} - 1\right)$$

($n$: toplam val penceresi, $N_u$: $u$'yu aşan sayısı.) Az veriyle bile kuyruk **modelden** geldiği için
Q99'un "val'deki şansa bağlı en kötü uçuş" kararsızlığını taşımaz.

## 4. Protokol

ML-1 ile birebir aynı: uçuş bazlı split (manifest), train=normal-only pencereler, 5 seed ortalama±std,
uçuş skoru = pencere skorlarının maksimumu, karşılaştırma tablosu IF-füzyon vs Dense AE vs LSTM-AE.


In [1]:
import json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from scipy.stats import genpareto
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

FEAT = ROOT / "data/gold/ml_features"
manifest = json.loads((FEAT / "split_manifest.json").read_text(encoding="utf-8"))

from src.ml.data.scaling import apply_scaler_params
from src.ml.data.windowing import build_windows

torch.set_num_threads(4)
NORMALS = {"normal", "benign"}

# ML-1 moduler feature'larin birlesimi -- AE girdisi (dar ve bilgilendirici)
AE_FEATURES = {
    "alfa": ["abs_roll_error", "abs_pitch_error", "abs_yaw_error", "roll_error_rate",
              "pitch_error_rate", "yaw_error_rate", "roll_error_2s_rms", "roll_error_5s_rms",
              "pitch_error_2s_rms", "pitch_error_5s_rms", "yaw_error_5s_rms",
              "roll_spec_energy_5s", "pitch_spec_energy_5s", "turn_residual",
              "alt_error", "aspd_error", "xtrack_error", "path_dev_mag", "climb_residual",
              "energy_rate", "altitude_rate", "abs_airspeed_error", "gps_speed_calc_mps"],
    "uav_attack": ["gps_step_m", "log_gps_speed", "gps_accel_mps2", "vertical_rate_calc",
                    "gps_speed_residual", "vertical_rate_residual", "course_change_deg",
                    "gps_frozen_count", "jamming_indicator", "noise_per_ms", "hdop", "vdop",
                    "satellites_used", "s_variance_m_s", "eph", "epv",
                    "roll_rate", "pitch_rate", "yaw_rate",
                    "attitude_missing", "num_missing_groups", "attitude_stale_s"],
}
WINDOW = {"alfa": 40, "uav_attack": 50}   # ~10 sn (4 Hz / 5 Hz)
STRIDE = {"alfa": 4, "uav_attack": 5}     # ~1 sn
MAX_GAP = {"alfa": 2.0, "uav_attack": 2.0}

def load_windows(name):
    df = pd.read_parquet(FEAT / name / f"{name}_ml_features.parquet")
    if name == "alfa":
        df = df[df["label"] != "unknown"]
    scaler = json.loads((ROOT / "artifacts/scalers" / f"{name}_robust_scaler.json").read_text())
    cols = [c for c in AE_FEATURES[name] if c in df.columns]
    scaled = apply_scaler_params(df, scaler)
    # apply_scaler_params impute etti -- maske icin orijinal NaN desenini kullan
    mask_df = df[cols].notna()
    scaled_vals = scaled[["source_id", "t_rel_s", "label"]].copy()
    for c in cols:
        scaled_vals[c] = scaled[c].where(mask_df[c])  # imputlanani geri NaN yap: maske pencereye tasinsin
    X, M, meta = build_windows(scaled_vals, cols, window=WINDOW[name],
                               stride=STRIDE[name], max_gap_s=MAX_GAP[name])
    print(f"{name}: {len(X)} pencere ({WINDOW[name]} x {len(cols)}), anomali orani {meta['is_anomaly'].mean():.2f}")
    return X, M, meta, cols

WIN = {name: load_windows(name) for name in ["alfa", "uav_attack"]}

alfa: 4421 pencere (40 x 23), anomali orani 0.89


uav_attack: 15741 pencere (50 x 22), anomali orani 0.69


## 5. Modeller ve maskeli kayıp

In [2]:
class DenseAE(nn.Module):
    def __init__(self, w, f):
        super().__init__()
        d = w * f
        self.enc = nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Linear(64, 16), nn.ReLU())
        self.dec = nn.Sequential(nn.Linear(16, 64), nn.ReLU(), nn.Linear(64, d))
        self.w, self.f = w, f

    def forward(self, x):                      # x: (b, w, f)
        z = self.enc(x.flatten(1))
        return self.dec(z).view(-1, self.w, self.f)


class LSTMAE(nn.Module):
    def __init__(self, f, hidden=32, latent=16):
        super().__init__()
        self.encoder = nn.LSTM(f, hidden, batch_first=True)
        self.to_latent = nn.Linear(hidden, latent)
        self.from_latent = nn.Linear(latent, hidden)
        self.decoder = nn.LSTM(hidden, hidden, batch_first=True)
        self.head = nn.Linear(hidden, f)

    def forward(self, x):                      # x: (b, w, f)
        _, (h, _) = self.encoder(x)
        z = self.to_latent(h[-1])              # (b, latent) -- pencerenin ozeti
        rep = self.from_latent(z).unsqueeze(1).repeat(1, x.shape[1], 1)  # RepeatVector
        dec, _ = self.decoder(rep)
        return self.head(dec)


def masked_mse(x, xhat, m, per_sample=False):
    """Maske-agirlikli MSE: eksik hucreler kayba girmez (H1 dersi)."""
    se = ((x - xhat) ** 2) * m
    if per_sample:
        return se.sum(dim=(1, 2)) / m.sum(dim=(1, 2)).clamp(min=1.0)
    return se.sum() / m.sum().clamp(min=1.0)


def train_ae(model, Xtr, Mtr, Xva, Mva, *, seed, epochs=40, batch=64, lr=1e-3, patience=5):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xtr_t, Mtr_t = torch.tensor(Xtr), torch.tensor(Mtr)
    Xva_t, Mva_t = torch.tensor(Xva), torch.tensor(Mva)
    best, best_state, bad = np.inf, None, 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr_t))
        for i in range(0, len(perm), batch):
            idx = perm[i : i + batch]
            x, m = Xtr_t[idx], Mtr_t[idx]
            loss = masked_mse(x, model(x), m)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = float(masked_mse(Xva_t, model(Xva_t), Mva_t))
        if vl < best - 1e-5:
            best, best_state, bad = vl, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    return model, best, ep + 1


def window_scores(model, X, M, batch=512):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            x = torch.tensor(X[i : i + batch]); m = torch.tensor(M[i : i + batch])
            out.append(masked_mse(x, model(x), m, per_sample=True).numpy())
    return np.concatenate(out) if out else np.array([])

print("model tanimlari hazir")

model tanimlari hazir


## 6. POT eşiği (GPD kuyruk modeli)

In [3]:
def pot_threshold(val_scores, q=1e-3, init_quantile=0.98):
    """Siffer et al. (KDD'17) POT: u = Q98(val), asimlara GPD fit, risk-q esigi."""
    u = np.quantile(val_scores, init_quantile)
    exceed = val_scores[val_scores > u] - u
    if len(exceed) < 5:
        return float(np.quantile(val_scores, 0.99)), "fallback_q99"
    xi, _, sigma = genpareto.fit(exceed, floc=0.0)
    n, Nu = len(val_scores), len(exceed)
    if abs(xi) < 1e-6:
        tau = u + sigma * np.log(Nu / (q * n))
    else:
        tau = u + (sigma / xi) * ((q * n / Nu) ** (-xi) - 1)
    return float(tau), f"GPD(xi={xi:.3f}, sigma={sigma:.4f})"

print("POT hazir")

POT hazir


## 7. Deney döngüsü — 5 seed × 2 kaynak × 2 mimari

In [4]:
def run_experiment(name, arch):
    X, M, meta, cols = WIN[name]
    labels = manifest["sources"][name]["flight_labels"]
    rows = []
    for split_name, split in manifest["sources"][name]["splits"].items():
        seed = split["seed"]
        tr_i = meta["source_id"].isin(split["train"]).to_numpy()
        va_i = meta["source_id"].isin(split["val"]).to_numpy()
        te_i = meta["source_id"].isin(split["test"]).to_numpy()
        w, f = X.shape[1], X.shape[2]
        model = DenseAE(w, f) if arch == "dense" else LSTMAE(f)
        t0 = time.time()
        model, vloss, eps = train_ae(model, X[tr_i], M[tr_i], X[va_i], M[va_i], seed=seed)
        s_va = window_scores(model, X[va_i], M[va_i])
        s_te = window_scores(model, X[te_i], M[te_i])
        tau_q99 = float(np.quantile(s_va, 0.99))
        tau_pot, pot_info = pot_threshold(s_va)
        te_meta = meta[te_i].assign(score=s_te)
        fs = te_meta.groupby("source_id")["score"].max()
        yf = np.array([0 if labels[sid] == "normal" else 1 for sid in fs.index])
        row = {"split": split_name, "epochs": eps, "sure_s": round(time.time() - t0, 1),
               "ucus_roc": roc_auc_score(yf, fs.values),
               "pencere_roc": roc_auc_score(te_meta["is_anomaly"], s_te)}
        for tname, tau in [("q99", tau_q99), ("pot", tau_pot)]:
            det = float((fs[yf == 1] > tau).mean())
            fa = float((fs[yf == 0] > tau).mean()) if (yf == 0).any() else np.nan
            row[f"tespit@{tname}"] = det
            row[f"yanlis_alarm@{tname}"] = fa
        row["pot_info"] = pot_info
        rows.append(row)
    return pd.DataFrame(rows).set_index("split")

results = {}
for name in ["alfa", "uav_attack"]:
    for arch in ["dense", "lstm"]:
        key = f"{name}/{arch}"
        results[key] = run_experiment(name, arch)
        r = results[key]
        print(f"=== {key} ===")
        num = r.drop(columns=["pot_info"])
        print((num.mean().round(3).astype(str) + " +- " + num.std().round(3).astype(str)).to_string(), "\n")

=== alfa/dense ===
epochs              35.8 +- 9.391
sure_s              0.88 +- 1.023
ucus_roc            0.622 +- 0.23
pencere_roc         0.316 +- 0.12
tespit@q99          0.55 +- 0.299
yanlis_alarm@q99     0.5 +- 0.354
tespit@pot          0.55 +- 0.299
yanlis_alarm@pot     0.5 +- 0.354 



=== alfa/lstm ===
epochs              23.4 +- 13.465
sure_s               1.32 +- 0.716
ucus_roc            0.731 +- 0.153
pencere_roc         0.282 +- 0.122
tespit@q99          0.689 +- 0.275
yanlis_alarm@q99      0.5 +- 0.354
tespit@pot          0.689 +- 0.275
yanlis_alarm@pot      0.5 +- 0.354 



=== uav_attack/dense ===
epochs                30.6 +- 8.05
sure_s                4.9 +- 1.439
ucus_roc            0.677 +- 0.167
pencere_roc         0.311 +- 0.181
tespit@q99          0.862 +- 0.084
yanlis_alarm@q99      0.8 +- 0.447
tespit@pot          0.815 +- 0.042
yanlis_alarm@pot      0.8 +- 0.447 



=== uav_attack/lstm ===
epochs              24.8 +- 16.162
sure_s              13.22 +- 7.736
ucus_roc            0.677 +- 0.126
pencere_roc          0.315 +- 0.18
tespit@q99          0.831 +- 0.064
yanlis_alarm@q99      0.8 +- 0.447
tespit@pot          0.785 +- 0.034
yanlis_alarm@pot      0.8 +- 0.447 



## 8. Karşılaştırma: IF-füzyon (ML-1) vs Dense AE vs LSTM-AE

In [5]:
ML1_BASELINE = {  # notebooks/02 sonuclarindan (5 seed ort.)
    "alfa": {"ucus_roc": 0.833, "tespit": 0.778, "yanlis_alarm": 0.30},
    "uav_attack": {"ucus_roc": 0.600, "tespit": 0.569, "yanlis_alarm": 0.40},
}
rows = []
for name in ["alfa", "uav_attack"]:
    rows.append({"kaynak": name, "model": "IF-fuzyon (ML-1)", **ML1_BASELINE[name]})
    for arch, mlabel in [("dense", "Dense AE"), ("lstm", "LSTM-AE")]:
        r = results[f"{name}/{arch}"]
        rows.append({"kaynak": name, "model": mlabel,
                     "ucus_roc": round(r["ucus_roc"].mean(), 3),
                     "tespit": round(r["tespit@pot"].mean(), 3),
                     "yanlis_alarm": round(r["yanlis_alarm@pot"].mean(), 3)})
comp = pd.DataFrame(rows).set_index(["kaynak", "model"])
print(comp.to_string())
print("\nEsik karsilastirmasi (Q99 vs POT, LSTM-AE):")
for name in ["alfa", "uav_attack"]:
    r = results[f"{name}/lstm"]
    print(f"{name}: tespit q99={r['tespit@q99'].mean():.2f} pot={r['tespit@pot'].mean():.2f} | "
          f"yanlis alarm q99={r['yanlis_alarm@q99'].mean():.2f} pot={r['yanlis_alarm@pot'].mean():.2f}")

                             ucus_roc  tespit  yanlis_alarm
kaynak     model                                           
alfa       IF-fuzyon (ML-1)     0.833   0.778           0.3
           Dense AE             0.622   0.550           0.5
           LSTM-AE              0.731   0.689           0.5
uav_attack IF-fuzyon (ML-1)     0.600   0.569           0.4
           Dense AE             0.677   0.815           0.8
           LSTM-AE              0.677   0.785           0.8

Esik karsilastirmasi (Q99 vs POT, LSTM-AE):
alfa: tespit q99=0.69 pot=0.69 | yanlis alarm q99=0.50 pot=0.50
uav_attack: tespit q99=0.83 pot=0.78 | yanlis alarm q99=0.80 pot=0.80


In [6]:
# tip bazinda LSTM-AE tespiti (POT esigi, seed ortalamasi)
def per_type_lstm(name):
    X, M, meta, _ = WIN[name]
    labels = manifest["sources"][name]["flight_labels"]
    accum = []
    for split_name, split in manifest["sources"][name]["splits"].items():
        tr_i = meta["source_id"].isin(split["train"]).to_numpy()
        va_i = meta["source_id"].isin(split["val"]).to_numpy()
        te_i = meta["source_id"].isin(split["test"]).to_numpy()
        model = LSTMAE(X.shape[2])
        model, _, _ = train_ae(model, X[tr_i], M[tr_i], X[va_i], M[va_i], seed=split["seed"])
        tau, _ = pot_threshold(window_scores(model, X[va_i], M[va_i]))
        te_meta = meta[te_i].assign(score=window_scores(model, X[te_i], M[te_i]))
        fs = te_meta.groupby("source_id").agg(score=("score", "max"), label=("label", lambda x: x.mode().iloc[0]))
        fs["flabel"] = [labels[sid] for sid in fs.index]
        for lbl, g in fs[fs["flabel"] != "normal"].groupby("flabel"):
            accum.append({"label": lbl, "tespit": float((g["score"] > tau).mean())})
    return pd.DataFrame(accum).groupby("label")["tespit"].mean().round(2)

print("ALFA LSTM-AE tip bazinda tespit@pot:"); print(per_type_lstm("alfa").to_string())
print("\nUAV Attack LSTM-AE tip bazinda tespit@pot:"); print(per_type_lstm("uav_attack").to_string())

ALFA LSTM-AE tip bazinda tespit@pot:


label
aileron_fault           0.66
aileron_rudder_fault    0.20
elevator_fault          0.60
engine_fault            0.76
rudder_fault            0.27

UAV Attack LSTM-AE tip bazinda tespit@pot:


label
gps_jamming     1.00
gps_spoofing    1.00
ping_dos        0.53


## 9. Sonuç

- Dense AE pipeline'ı doğruladı (debug amacına ulaştı); LSTM-AE zamansal bağlamla asıl karşılaştırma modeli.
- POT eşiği, Q99'un az-val-verisi kararsızlığına (H2) karşı ilkesel alternatif — iki eşik yan yana raporlandı.
- Maske-ağırlıklı kayıp, impute'un anomali gizlemesini (H1) mimari düzeyde çözdü.
- Karşılaştırma tablosu hangi ailelerde sequence modellemenin IF-füzyonu geçtiğini gösteriyor;
  geçemediği yerlerde neden (kısa senaryolar, az normal veri) `docs/ML1_BULGULAR_VE_HATALAR.md` H6 ile tutarlı.

**ML-3**: ablation matrisi (missingness±, airspeed±, residual-only), sentetik enjeksiyon test seti,
SEAD τ-kalibrasyonlu transfer, USAD/TranAD denemesi (bkz. `docs/ANOMALI_METOT_ARASTIRMASI.md`).
